# 1 교사 점수 · 채택 관문 — B+(mn10 교사) 대 B0(조각 교사)

**L4 로 실행하십시오.** [셀 1] 이 Blackwell(sm_100·sm_120)이면 멈춥니다. GPU 가 없으면 CPU 로 돕니다(시간 제한 없음).

**'모두 실행' 한 번으로 끝까지 돕니다.** Mark4.1~4.8 교사 노트북과 동시에 시작해도 됩니다 — [셀 5] 가 교사 16개(B+ 8 · B0 8)가 지금 데이터로 끝날 때까지 기다립니다.

| 순서 | 하는 일 | 명령 |
|---|---|---|
| 1 | 교사 대기 | 체크포인트에 적힌 데이터 인덱스 해시가 지금 Mark4.x 인덱스와 같고, B+ 학습 후 판정이 '통과'인지 확인 |
| 2 | 반입 | `model/import_teachers.py --kind mn10` · `--kind segment` → `resources/` · `verify_release.py` |
| 3 | 점수 캐시 | `model/make_teacher_scores.py --teacher_kind mn10` · `segment` → `plots/teacher_scores_<kind>.{csv,json}` |
| 4 | 채택 관문 | `model/teacher_gate_report.py --candidate plots/teacher_scores_mn10.csv --reference plots/teacher_scores_segment.csv` |

- **기준 교사는 B0 입니다(2026-10-23 결정).** 같은 새 데이터로 학습한 판이라 공정합니다. 3차 교사 값은 관문 출력에 참고로만 찍힙니다.
- 관문 네 조건(C1 val 430 Macro F1 · C2 정답 2,000 Macro F1 · C3 결합기 5겹 교차검증 · C4 남의 클래스 손들기)을 **모두 넘으면 '채택(B+)'** 이고, 다음 노트북(`2 Mark5.3 본실험 80회`)이 자동으로 mn10 교사로 진행합니다.
- 하나도 못 넘거나 일부만 넘으면 다음 노트북이 멈춥니다. 결과를 보고 함께 정합니다(B0 가 뽑히면 `_TC` 아블레이션 처리도 그때 정합니다).
- 점수 캐시는 지금 `resources/` 교사로 만든 것이면 건너뜁니다(해시 대조).

In [ ]:
#========================================================================
# [셀 1] 런타임 확인 — L4 권장 · Blackwell(sm_100·120)이면 멈춤 · CPU 는 허용(멈추지 않음)
#========================================================================
import subprocess, sys
_probe = subprocess.run([sys.executable, "-c",
    "import os, torch\n"
    "ok = torch.cuda.is_available()\n"
    "cap = torch.cuda.get_device_capability(0) if ok else None\n"
    "print('GPU', torch.cuda.get_device_name(0) if ok else '없음 — CPU 로 돈다')\n"
    "print('SM', cap[0] * 10 + cap[1] if cap else 'cpu')\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| CPU 코어', os.cpu_count())"],
    capture_output=True, text=True)
print(_probe.stdout, _probe.stderr[-2000:])
_sm = next((l.split()[1] for l in _probe.stdout.splitlines() if l.startswith("SM ")), "cpu")
if _sm in ("100", "120"):
    raise SystemExit("[STOP] Blackwell(sm_100/sm_120) 런타임입니다. L4 로 바꿔 다시 '모두 실행' 하십시오(2026-10-17 G4 계산 오류).")
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv 2>/dev/null || echo "nvidia-smi 없음(CPU 런타임)"
!free -g | head -2

In [ ]:
#========================================================================
# [셀 2] 구글 드라이브 마운트
#========================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#========================================================================
# [셀 3] 작업 폴더 · 데이터 목록 · 드라이브 코드가 2026-10-25 판인가
#========================================================================
import csv, hashlib, json, os, re, time
EXP = "/content/drive/MyDrive/NewProject/codes/experiments"
ROOT = f"{EXP}/Mark5.3"
if not os.path.isdir(ROOT):
    raise SystemExit(f"[STOP] 코드 폴더가 없습니다: {ROOT}")
os.chdir(ROOT)
sys.path.insert(0, f"{ROOT}/model")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


for name, n_expected in {"dataset_labeled.csv": 2000, "dataset_val.csv": 430,
                         "dataset_unlabeled.csv": 2000, "dataset_test.csv": 430}.items():
    if not os.path.isfile(name):
        raise SystemExit(f"[STOP] 데이터 목록이 없습니다: {name}")
    with open(name, newline="", encoding="utf-8-sig") as f:
        n = sum(1 for _ in csv.DictReader(f))
    print(name, n)
    if n != n_expected:
        raise SystemExit(f"[STOP] {name} 행 수 {n}, 기대 {n_expected}")
NEED = {"model/import_teachers.py": ['choices=["segment", "mn10"]'],
        "model/make_teacher_scores.py": ['"--teacher_kind"', "mark53_audio_sha256"],
        "model/teacher_gate_report.py": ["C4_raise_off_mean"],
        "model/make_combiner_labels.py": ['"--scores"'],
        "model/label_freshness.py": ["def check_scores", "def check_pseudo_csv", "def audio_manifest"],
        "model/run_experiments.py": ["CMB_PLAN = [", 'choices=["paper", "plsource", "pl4", "cmb"]'],
        "model/aggregate_results.py": ['"--main_pseudo_source"'],
        "model/make_paper_tables.py": ['"--main_pseudo_source"', '"plc"'],
        "model/verify_release.py": ['"--refresh_source"']}
miss = [(f, s) for f, ss in NEED.items() for s in ss
        if not os.path.isfile(f) or s not in open(f, encoding="utf-8").read()]
if miss:
    raise SystemExit(f"[STOP] 드라이브 코드가 10-25 판이 아닙니다: {miss}")
size = os.path.getsize("weights/mn10_as_mAP_471.pt") if os.path.isfile("weights/mn10_as_mAP_471.pt") else None
print("mn10 가중치", size, "bytes (기대 19708753)")
if size != 19708753:
    raise SystemExit("[STOP] weights/mn10_as_mAP_471.pt 가 없거나 크기가 다릅니다")
print("코드 판 확인 완료")

In [ ]:
#========================================================================
# [셀 4] 패키지 — 코랩의 GPU torch·torchvision 은 그대로 둔다 · 문장 인코더 네 패키지만 판 고정
#========================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "sentence-transformers==2.7.0", "transformers==4.46.3",
                "huggingface-hub==0.36.2", "tokenizers==0.20.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "librosa==0.10.2.post1", "soundfile==0.12.1",
                "pydub==0.25.1", "thop==0.1.1.post2209072238"], check=True)
import sentence_transformers, transformers, huggingface_hub, tokenizers
got = {"sentence-transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
       "huggingface-hub": huggingface_hub.__version__, "tokenizers": tokenizers.__version__}
want = {"sentence-transformers": "2.7.0", "transformers": "4.46.3", "huggingface-hub": "0.36.2", "tokenizers": "0.20.3"}
print(got)
if got != want:
    raise SystemExit(f"[STOP] 패키지 판 불일치 {got} — 런타임 → 세션 다시 시작 → 다시 '모두 실행'")
import shutil
from pathlib import Path
HF_SRC = Path("/content/drive/MyDrive/NewProject/hf_cache/models--sentence-transformers--all-MiniLM-L6-v2")
HF_HUB = Path.home() / ".cache" / "huggingface" / "hub"
if not HF_SRC.is_dir():
    raise SystemExit(f"[STOP] 문장 인코더 캐시가 없습니다: {HF_SRC}")
HF_HUB.mkdir(parents=True, exist_ok=True)
shutil.copytree(HF_SRC, HF_HUB / HF_SRC.name, dirs_exist_ok=True)
os.environ.update({"HF_HOME": str(HF_HUB.parent), "HF_HUB_CACHE": str(HF_HUB),
                   "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
from sentence_transformers import SentenceTransformer
SentenceTransformer("all-MiniLM-L6-v2")
print("문장 인코더 오프라인 캐시 준비 완료")

In [ ]:
#========================================================================
# [셀 5] 교사 16개(B+ 8 · B0 8)가 지금 데이터로 끝날 때까지 기다린다 — 시간 제한 없음 · B+ 판정 불합격이면 멈춘다
#========================================================================
import torch
TARGETS = ["heavy_impact", "dragging", "construction", "machine_noise", "talking", "water_toilet", "water_shower", "dog_bark"]
_cache = {}


def _load(path, **kw):
    key = (path, os.path.getmtime(path), os.path.getsize(path))
    if key not in _cache:
        _cache[key] = torch.load(path, map_location="cpu", **kw)
    return _cache[key]


def teacher_state(k):
    repo, mv, target = f"{EXP}/Mark4.{k}", f"mark4.{k}", TARGETS[k - 1]
    index_sha = sha256(f"{repo}/dataset_index_{mv}.csv")
    # 교사 노트북이 감사판 오디오(data_release_mark4.x.json)를 대조한 뒤 학습한 판만 받는다 — 인덱스는 이름만 담아 옛 오디오를 못 가린다
    release = json.load(open(f"{repo}/data_release_{mv}.json", encoding="utf-8"))["release_sha256"]
    mk_path = f"{repo}/plots/teacher_data_release_{mv}.json"
    mk = json.load(open(mk_path, encoding="utf-8")) if os.path.isfile(mk_path) else {}
    mk = mk if mk.get("release_sha256") == release else {}
    ck, ev = f"{repo}/model/mn10_teacher_{mv}.pt", f"{repo}/plots/mn10_teacher_eval_{mv}.json"
    if not (os.path.isfile(ck) and os.path.isfile(ev)):
        bp = "파일 없음"
    else:
        prov = _load(ck, weights_only=False).get("provenance") or {}
        e = json.load(open(ev, encoding="utf-8"))
        if prov.get("dataset_index_sha256") != index_sha:
            bp = "옛 데이터 판"
        elif (e.get("provenance") or {}).get("run_id") != prov.get("run_id"):
            bp = "판정 json 이 다른 회차"
        elif not e["verdict"]["pass"]:
            bp = "판정 불합격"
        elif mk.get("bplus_run_id") != prov.get("run_id"):
            bp = "데이터 판 표식 없음"
        else:
            bp = "완료"
    enc, clf = f"{repo}/model/best_teacher_encoder_{mv}.pth", f"{repo}/model/best_teacher_classifier_{mv}.pth"
    met = f"{repo}/plots/teacher_val_metrics_{mv}.json"
    if not all(os.path.isfile(p) for p in (enc, clf, met)):
        b0 = "파일 없음"
    else:
        ep, cp = (_load(p).get("provenance") or {} for p in (enc, clf))
        if ep != cp:
            b0 = "encoder·classifier provenance 다름"
        elif ep.get("dataset_index_sha256") != index_sha:
            b0 = "옛 데이터 판"
        elif ep.get("train_objective") != "segment" or ep.get("classes") != [target, "others"]:
            b0 = "recipe run3 판이 아님"
        elif json.load(open(met, encoding="utf-8")).get("teacher_run_id") != ep.get("run_id"):
            b0 = "val 지표가 다른 회차"
        elif mk.get("b0_run_id") != ep.get("run_id"):
            b0 = "데이터 판 표식 없음"
        else:
            b0 = "완료"
    return bp, b0


t0 = time.time()
while True:
    states = {k: teacher_state(k) for k in range(1, 9)}
    failed = [k for k, (bp, _) in states.items() if bp == "판정 불합격"]
    if failed:
        raise SystemExit(f"[STOP] B+ 학습 후 판정 불합격: Mark4.{failed} — 그 교사 노트북의 마지막 셀 출력을 보고 정한 뒤 다시")
    left = {k: s for k, s in states.items() if s != ("완료", "완료")}
    if not left:
        break
    print(time.strftime("%H:%M:%S"), f"(경과 {(time.time() - t0) / 60:.0f}분) 남은 교사 {len(left)}/8:",
          " · ".join(f"4.{k} B+ {bp} / B0 {b0}" for k, (bp, b0) in left.items()), flush=True)
    time.sleep(60)
print("교사 16개가 모두 지금 데이터로 끝났습니다.")

In [ ]:
#========================================================================
# [셀 6] 반입 — mn10 교사 8개 · 조각 교사 16파일을 resources/ 로(원자적 교체) → release 검사
#========================================================================
for kind in ("mn10", "segment"):
    subprocess.run([sys.executable, "-u", "model/import_teachers.py", "--kind", kind, "--source_root", EXP], check=True)
subprocess.run([sys.executable, "model/verify_release.py"], check=True)
subprocess.run([sys.executable, "tests/verify_teacher_release_contract.py"], check=True)
for name in ("teacher_manifest_mn10.json", "teacher_manifest.json"):
    m = json.load(open(f"resources/{name}", encoding="utf-8"))
    print(f"{name}: kind {m.get('kind')} · 파일 {len(m['files'])}개 · release {m['teacher_release_sha256'][:12]}")

In [ ]:
#========================================================================
# [셀 7] 교사 점수 캐시 — 지금 resources/ 교사로 만든 캐시면 건너뛴다(해시 대조)
#========================================================================
import label_freshness


def scores_fresh(kind):
    csv_path = f"plots/teacher_scores_{kind}.csv"
    try:
        label_freshness.check_scores(csv_path)
        if kind == "mn10":
            label_freshness.check_features(f"plots/teacher_feats_{kind}.pt", csv_path)
        return True
    except (label_freshness.StaleFileError, FileNotFoundError) as e:
        print(f"  {kind}: 다시 만듭니다 — {e}")
        return False


for kind in ("mn10", "segment"):
    if scores_fresh(kind):
        print(f"{kind}: 지금 교사의 점수 캐시가 있습니다 — 건너뜁니다.")
        continue
    subprocess.run([sys.executable, "-u", "model/make_teacher_scores.py", "--teacher_kind", kind], check=True)
    if not scores_fresh(kind):
        raise SystemExit(f"[STOP] {kind} 점수 캐시를 만든 뒤에도 최신 판 확인을 통과하지 못했습니다")
for kind in ("mn10", "segment"):
    print(f"plots/teacher_scores_{kind}.csv  SHA-256 {sha256(f'plots/teacher_scores_{kind}.csv')[:12]}")

In [ ]:
#========================================================================
# [셀 8] 채택 관문 — 후보 B+(mn10) 대 기준 B0(segment)
#========================================================================
subprocess.run([sys.executable, "-u", "model/teacher_gate_report.py",
                "--candidate", "plots/teacher_scores_mn10.csv", "--reference", "plots/teacher_scores_segment.csv"], check=True)
GATE = "plots/teacher_gate_mn10_vs_segment.json"
g = json.load(open(GATE, encoding="utf-8"))
print()
for k, v in g["conditions"].items():
    print(f"  {k}: {'만족' if v else '불만족'}")
print(f"[판정] {g['n_satisfied']}/4 → {g['verdict']}")
if g["verdict"] == "채택(B+)":
    print("다음: '2 Mark5.3 본실험 80회' 노트북이 mn10 교사로 진행합니다.")
else:
    print("다음: '2 Mark5.3 본실험 80회' 노트북은 이 판정에서 멈춥니다. 위 지표를 보고 어느 교사를 쓸지 함께 정합니다.")